# PathSnap exploratory data analysis
Run from the repository root or `notebooks/` after installing PathSnap. The inventory verifies both supplied partitions. All analytical sections use training trips. Run `python scripts/build_dataset.py` to make the point-level alignment Parquet available.


## Trip inventory · train and student_test

The inventory is the only section that reads both supplied partitions. Each expected file is checked before parsing, and IDs are kept as strings.

In [ ]:

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathsnap import config
from pathsnap.io import list_trips, load_trip

EDA = config.RESULTS_DIR / "eda"
EDA.mkdir(parents=True, exist_ok=True)


def save_table(table, name):
    table.to_csv(EDA / f"{name}.csv", index=False)
    return table


def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(EDA / f"{name}.png", dpi=150, facecolor="white")
    display(fig)
    plt.close(fig)


def trip_inventory(split, root):
    records, tracks = [], {}
    expected_files = ("track", "nodes", "arcs", "route")
    for folder in list_trips(root):
        files = {ext: folder / f"{folder.name}.{ext}" for ext in expected_files}
        missing = [ext for ext, path in files.items() if not path.is_file()]
        if missing:
            raise FileNotFoundError(f"{split}/{folder.name}: missing {', '.join(missing)}")
        trip = load_trip(folder)
        tracks[trip.trip_id] = trip.track if split == "train" else None
        records.append({
            "split": split, "trip_id": trip.trip_id,
            "files_present": " ".join(expected_files),
            **{f"{ext}_bytes": path.stat().st_size for ext, path in files.items()},
            "n_points": trip.n_points, "n_nodes": len(trip.nodes),
            "n_arcs": len(trip.arcs), "n_route": len(trip.route),
        })
    return pd.DataFrame.from_records(records), tracks


train_inventory, train_tracks = trip_inventory("train", config.TRAIN_DIR)
student_inventory, _ = trip_inventory("student_test", config.STUDENT_TEST_DIR)
inventory = pd.concat([train_inventory, student_inventory], ignore_index=True)
counts = inventory.groupby("split", as_index=False).agg(
    trips=("trip_id", "size"), gps_points=("n_points", "sum"),
    min_points=("n_points", "min"), median_points=("n_points", "median"),
    max_points=("n_points", "max"),
)
save_table(inventory, "trip_inventory")
display(save_table(counts, "trip_inventory_summary"))

## GPS data quality · train

Raw longitude, latitude, and timestamp fields are checked independently. A repeated position can reflect a stop, while a duplicate row has the same position and timestamp.

In [ ]:
gps_fields = ["lon", "lat", "t"]
quality_rows = []
for trip_id, track in train_tracks.items():
    gps = track[gps_fields]
    times = gps["t"].dropna()
    quality_rows.append({
        "split": "train", "trip_id": trip_id,
        **{f"{col}_dtype": str(gps[col].dtype) for col in gps_fields},
        **{f"{col}_missing": int(gps[col].isna().sum()) for col in gps_fields},
        "infinite_fields": int(np.isinf(gps.to_numpy()).sum()),
        "out_of_range_coordinates": int(
            ((gps["lon"].abs() > 180) | (gps["lat"].abs() > 90)).sum()
        ),
        "duplicate_rows": int(gps.duplicated().sum()),
        "consecutive_repeated_positions": int(track["is_dup"].sum()),
        "loader_jump_flags": int(track["is_jump"].sum()),
        "fractional_timestamps": int((times % 1 != 0).sum()),
    })

quality = save_table(pd.DataFrame(quality_rows), "gps_quality")
display(quality.select_dtypes(include="number").sum().to_frame("train total"))

## GPS sampling and gaps · train

Intervals use adjacent observations in file order. The distribution keeps the tail on logarithmic axes. Nonpositive intervals are counted separately, and the first observation in each trip has no preceding interval.

In [ ]:
intervals = pd.concat(
    [
        track[["t", "dt_prev"]].assign(split="train", trip_id=trip_id, point_idx=track.index)
        for trip_id, track in train_tracks.items()
    ],
    ignore_index=True,
).dropna(subset=["dt_prev"])
time_summary = intervals.groupby(["split", "trip_id"], as_index=False).agg(
    n_intervals=("dt_prev", "size"),
    nonpositive=("dt_prev", lambda values: (values <= 0).sum()),
    share_1s=("dt_prev", lambda values: np.isclose(values, 1, rtol=0, atol=1e-8).mean()),
    count_over_5s=("dt_prev", lambda values: (values > config.GAP_S).sum()),
    max_interval_s=("dt_prev", "max"),
)
save_table(time_summary, "sampling_by_trip")
display(save_table(intervals.nlargest(10, "dt_prev"), "largest_gps_gaps"))

positive = intervals.loc[intervals["dt_prev"] > 0, "dt_prev"]
bins = np.geomspace(positive.min() * 0.99, positive.max() * 1.01, 55)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(positive, bins=bins, color="#207b91")
ax.axvline(config.GAP_S, color="#b34a3c", linestyle="--", label="Gap threshold")
ax.set(xscale="log", yscale="log", xlabel="Time since previous observation (s)",
       ylabel="GPS intervals (count)", title="GPS sampling intervals · train")
ax.legend()
save_figure(fig, "sampling_intervals")

## Geographic distribution · train

The extent CSV uses all training observations. Each map marker represents one training trip’s starting observation.

In [ ]:
import folium

extent_rows = []
world = folium.Map(tiles="CartoDB positron", zoom_start=2)
for trip_id, track in train_tracks.items():
    lon, lat = track["lon"], track["lat"]
    extent_rows.append({
        "split": "train", "trip_id": trip_id, "start_lon": lon.iloc[0], "start_lat": lat.iloc[0],
        "min_lon": lon.min(), "max_lon": lon.max(), "min_lat": lat.min(), "max_lat": lat.max(),
    })
    folium.CircleMarker(
        [lat.iloc[0], lon.iloc[0]], radius=4, color="#207b91", fill=True, fill_opacity=0.8,
        tooltip=f"Train trip {trip_id}",
    ).add_to(world)

extent = save_table(pd.DataFrame(extent_rows), "geographic_extent")
world.fit_bounds([[extent["start_lat"].min(), extent["start_lon"].min()],
                  [extent["start_lat"].max(), extent["start_lon"].max()]])
world_html = "\n".join(line.rstrip() for line in world.get_root().render().splitlines())
(EDA / "train_trip_starts.html").write_text(world_html + "\n", encoding="utf-8")
display(extent.agg({"min_lon": "min", "max_lon": "max", "min_lat": "min", "max_lat": "max"}))
display(world)

## Road-network topology · train

Each trip has a separate directed network. Component counts use the canonical sparse adjacency from RoadNetwork with all node IDs included. Degree counts include all canonical arcs.

In [ ]:
from scipy.sparse.csgraph import connected_components

from pathsnap.network import RoadNetwork

topology_rows = []
degree_rows = []
for folder in list_trips(config.TRAIN_DIR):
    trip = load_trip(folder, with_route=False)
    net = RoadNetwork(trip)
    canon_nonloops = net.is_canon & (net.u != net.v)
    n_weak, weak_labels = connected_components(net.graph, directed=True, connection="weak")
    n_strong, strong_labels = connected_components(net.graph, directed=True, connection="strong")
    topology_rows.append({
        "split": "train", "trip_id": trip.trip_id, "n_nodes": net.n_nodes, "n_arcs": net.n_arcs,
        "duplicate_directed_arcs": int((~net.is_canon).sum()),
        "self_loops": int((net.u == net.v).sum()),
        "zero_length_nonloops": int(((net.u != net.v) & (net.length == 0)).sum()),
        "reverse_twin_share": float((net.rev_of[canon_nonloops] >= 0).mean()),
        "isolated_nodes": int(((net.in_deg + net.out_deg) == 0).sum()),
        "weak_components": n_weak, "strong_components": n_strong,
        "largest_weak_share": np.bincount(weak_labels).max() / net.n_nodes,
        "largest_strong_share": np.bincount(strong_labels).max() / net.n_nodes,
    })
    for direction, values in [("in", net.in_deg), ("out", net.out_deg)]:
        counts = np.bincount(values)
        degree_rows.extend(
            {"split": "train", "trip_id": trip.trip_id, "direction": direction,
             "degree": degree, "nodes": count}
            for degree, count in enumerate(counts) if count
        )

topology = save_table(pd.DataFrame(topology_rows), "network_topology")
degrees = save_table(pd.DataFrame(degree_rows), "network_degrees")
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, col, label in zip(axs, ["n_nodes", "n_arcs"], ["Nodes", "Directed arcs"]):
    ax.hist(topology[col], bins=np.geomspace(topology[col].min() * 0.99, topology[col].max() * 1.01, 23),
            color="#207b91")
    ax.set(xscale="log", xlabel=f"{label} per trip (count)", ylabel="Training trips (count)")
fig.suptitle("Road-network size · train")
save_figure(fig, "network_sizes")

fig, ax = plt.subplots(figsize=(8, 4))
for direction, group in degrees.groupby("direction"):
    totals = group.groupby("degree")["nodes"].sum()
    ax.plot(totals.index, totals.values, marker=".", label=f"{direction.title()}-degree")
ax.set(yscale="log", xlabel="Directed degree (arcs per node)", ylabel="Nodes (count)",
       title="Canonical road-network degree · train")
ax.legend()
save_figure(fig, "network_degree")

## Reference route lengths and breaks · train

Length sums projected arc lengths in reference order, including repeated traversals. A break occurs when the previous arc’s destination differs from the next arc’s origin.

In [ ]:
from pathsnap.geo import Projector

route_rows = []
for folder in list_trips(config.TRAIN_DIR):
    trip = load_trip(folder)
    endpoints = trip.arcs.iloc[trip.route]
    u, v = endpoints["u"].to_numpy(), endpoints["v"].to_numpy()
    proj = Projector.for_track(trip.track["lon"].to_numpy(), trip.track["lat"].to_numpy())
    x, y = proj.fwd(trip.nodes["lon"].to_numpy(), trip.nodes["lat"].to_numpy())
    route_rows.append({
        "split": "train", "trip_id": trip.trip_id, "n_route_arcs": len(u),
        "length_km": np.hypot(x[v] - x[u], y[v] - y[u]).sum() / 1000,
        "n_transitions": max(len(u) - 1, 0),
        "n_breaks": int((v[:-1] != u[1:]).sum()),
    })

routes = save_table(pd.DataFrame(route_rows), "reference_route_lengths")
display(routes.query("n_breaks > 0"))
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(routes["length_km"], bins=25, color="#207b91")
ax.set(xlabel="Reference route length (km)", ylabel="Training trips (count)",
       title="Reference route length · train")
save_figure(fig, "reference_route_lengths")